In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    cross_val_predict
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import confusion_matrix, classification_report, f1_score
import sklearn

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.8.0


In [2]:
rng = np.random.default_rng(42)
n = 1200

df = pd.DataFrame({
    # Numeriska features
    "age": rng.integers(18, 70, size=n).astype(float),
    "income": rng.normal(35000, 12000, size=n),
    "tenure_months": rng.integers(0, 120, size=n).astype(float),

    # Kategoriska features
    "city": rng.choice(["Stockholm", "Göteborg", "Malmö"], size=n, p=[0.45, 0.35, 0.20]),
    "plan": rng.choice(["basic", "plus", "pro"], size=n, p=[0.55, 0.30, 0.15]),
    "channel": rng.choice(["web", "store", "partner"], size=n, p=[0.6, 0.25, 0.15]),
})

df.loc[rng.random(n) < 0.06, "age"] = np.nan
df.loc[rng.random(n) < 0.10, "income"] = np.nan
df.loc[rng.random(n) < 0.04, "city"] = np.nan
df.loc[rng.random(n) < 0.05, "plan"] = np.nan

noise = rng.normal(0, 0.18, size=n)

score = (
    -0.045 * df["tenure_months"].fillna(df["tenure_months"].median())
    -0.00003 * df["income"].fillna(df["income"].median())
    +0.80 * (df["plan"].fillna("basic") == "basic").astype(int)
    +0.45 * (df["channel"] == "partner").astype(int)
    +0.30 * (df["city"].fillna("Stockholm") == "Malmö").astype(int)
    +0.40
    +noise
)

prob = 1 / (1 + np.exp(-score))
df["target_churn"] = (rng.random(n) < prob).astype(int)

df.head()

,age,income,tenure_months,city,plan,channel,target_churn
0,22.0,49975.399821,115.0,Göteborg,basic,store,0
1,58.0,31969.791988,63.0,Stockholm,plus,store,0
2,52.0,39361.452054,77.0,Malmö,plus,web,0
3,40.0,6080.936410,72.0,Göteborg,plus,web,0
4,40.0,NaN,119.0,Stockholm,pro,web,0


In [3]:
print("Shape:", df.shape)
print("\nMissing per column:\n", df.isna().sum())
print("\nTarget distribution:\n", df["target_churn"].value_counts(normalize=True))

Shape: (1200, 7)

Missing per column:
 age               74
income           109
tenure_months      0
city              55
plan              75
channel            0
target_churn       0
dtype: int64

Target distribution:
 target_churn
0    0.865
1    0.135
Name: proportion, dtype: float64


In [4]:
target_col = "target_churn"
X = df.drop(columns=[target_col])
y = df[target_col]

numeric_features = ["age", "income", "tenure_months"]
categorical_features = ["city", "plan", "channel"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", y_train.mean().round(3), "Test positive rate:", y_test.mean().round(3))

Train: (900, 6) Test: (300, 6)
Train positive rate: 0.136 Test positive rate: 0.133


In [5]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [6]:
lr = LogisticRegression(max_iter=500, random_state=42)
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
gb = GradientBoostingClassifier(random_state=42)

pipe_lr = Pipeline([("preprocess", preprocess), ("model", lr)])
pipe_rf = Pipeline([("preprocess", preprocess), ("model", rf)])
pipe_gb = Pipeline([("preprocess", preprocess), ("model", gb)])

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "f1"

In [8]:
scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring=SCORING)
print("CV scores (Logistic Regression):", scores_lr)
print("mean:", scores_lr.mean().round(3), "std:", scores_lr.std().round(3))

CV scores (Logistic Regression): [0.33333333 0.3125     0.28571429 0.25       0.28571429]
mean: 0.293 std: 0.028


In [9]:
baseline_rows = []

for name, pipe in [("LogisticRegression", pipe_lr), ("RandomForest", pipe_rf), ("GradientBoosting", pipe_gb)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING)
    baseline_rows.append({"model": name, "mean": scores.mean(), "std": scores.std()})

baseline_table = pd.DataFrame(baseline_rows).sort_values("mean", ascending=False)
baseline_table

,model,mean,std
2,GradientBoosting,0.365238,0.066356
0,LogisticRegression,0.293452,0.028147
1,RandomForest,0.267741,0.085930


In [10]:
top2_names = baseline_table["model"].head(2).tolist()
print("Top-2 selected:", top2_names)

Top-2 selected: ['GradientBoosting', 'LogisticRegression']


In [11]:
C_values = [0.1, 1.0, 10.0]

manual = []

for C in C_values:
    tmp = pipe_lr.set_params(model__C = C)

    scores = cross_val_score(tmp, X_train, y_train, cv=cv, scoring=SCORING)
    manual.append({"C": C, "mean": scores.mean(), "std": scores.std()})

pd.DataFrame(manual).sort_values("mean", ascending=False)

,C,mean,std
2,10.0,0.298701,0.018569
1,1.0,0.293452,0.028147
0,0.1,0.199523,0.081297


In [12]:
all_params_lr = pipe_lr.get_params()
all_keys_lr = list(all_params_lr.keys())
print("Total number of params in LogisticRegression pipeline:", len(all_keys_lr))

# for k in all_params_lr:
#     print(k)

filtered_lr = [k for k in all_keys_lr if k.startswith(("model__", "preprocess__"))]
# for k in filtered_lr:
#     print(k)

all_keys_gb = list(pipe_gb.get_params().keys())
filtered_gb = [k for k in all_keys_gb if k.startswith(("model__", "preprocess__"))]
# for k in filtered_gb:
#     print(k)

Total number of params in LogisticRegression pipeline: 65


In [16]:
param_grid_gb = {
    "model__n_estimators": [200, 400],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__subsample": [0.7, 1.0],
    "model__min_samples_leaf": [1, 10]
}

grid_gb = GridSearchCV(
    estimator=pipe_gb,
    param_grid=param_grid_gb,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_gb.fit(X_train, y_train)

print("Best GB params:", grid_gb.best_params_)
print("Best GB CV score:", round(grid_gb.best_score_, 3))

Best GB params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_samples_leaf': 1, 'model__n_estimators': 200, 'model__subsample': 1.0}
Best GB CV score: 0.385


In [21]:
res_gb = pd.DataFrame(grid_gb.cv_results_)
res_gb.sort_values("rank_test_score").head(8)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__learning_rate,param_model__max_depth,param_model__min_samples_leaf,param_model__n_estimators,param_model__subsample,params,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
25,0.203132,0.009588,0.005646,0.000419,0.05,3,1,200,1.0,"{'model__learning_rate': 0.05, 'model__max_dep...",...,0.385427,0.081532,1,0.812121,0.738854,0.797619,0.767296,0.814371,0.786052,0.028965
43,0.331811,0.058136,0.004198,0.000742,0.10,3,1,400,1.0,"{'model__learning_rate': 0.1, 'model__max_dept...",...,0.384673,0.083517,2,1.000000,0.994872,1.000000,1.000000,1.000000,0.998974,0.002051
15,0.424384,0.065151,0.006353,0.000839,0.03,3,10,400,1.0,"{'model__learning_rate': 0.03, 'model__max_dep...",...,0.381383,0.126930,3,0.748466,0.754717,0.800000,0.759494,0.743902,0.761316,0.020058
27,0.435968,0.042135,0.007173,0.001093,0.05,3,1,400,1.0,"{'model__learning_rate': 0.05, 'model__max_dep...",...,0.374173,0.110994,4,0.918033,0.893855,0.935484,0.928962,0.934783,0.922223,0.015500
11,0.408871,0.004777,0.005838,0.000397,0.03,3,1,400,1.0,"{'model__learning_rate': 0.03, 'model__max_dep...",...,0.367559,0.076765,5,0.845238,0.800000,0.834286,0.800000,0.812121,0.818329,0.018380
32,0.150876,0.013198,0.006274,0.000841,0.10,2,1,200,0.7,"{'model__learning_rate': 0.1, 'model__max_dept...",...,0.363816,0.105799,6,0.780488,0.692810,0.771084,0.738854,0.716981,0.740043,0.032752
29,0.235978,0.028180,0.008010,0.002356,0.05,3,10,200,1.0,"{'model__learning_rate': 0.05, 'model__max_dep...",...,0.360418,0.131117,7,0.753086,0.709677,0.743902,0.734177,0.753086,0.738786,0.016148
35,0.330205,0.064917,0.006138,0.001601,0.10,2,1,400,1.0,"{'model__learning_rate': 0.1, 'model__max_dept...",...,0.352301,0.085727,8,0.886364,0.837209,0.872928,0.892655,0.875000,0.872831,0.019234


In [23]:
param_grid_lr = {
    "model__C": [0.1, 1.0, 10.0],
    "model__solver": ["lbfgs", "liblinear"],
    "model__class_weight": [None, "balanced"],
    "preprocess__num__imputer__strategy": ["median", "mean"]
}

grid_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_lr.fit(X_train, y_train)

print("Best LR params:", grid_lr.best_params_)
print("Best LR CV score:", round(grid_lr.best_score_, 3))

Best LR params: {'model__C': 10.0, 'model__class_weight': 'balanced', 'model__solver': 'lbfgs', 'preprocess__num__imputer__strategy': 'mean'}
Best LR CV score: 0.482


In [24]:
res_lr = pd.DataFrame(grid_lr.cv_results_)
res_lr.sort_values("rank_test_score").head(8)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__C,param_model__class_weight,param_model__solver,param_preprocess__num__imputer__strategy,params,split0_test_score,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
21,0.008861,0.000517,0.004700,0.000375,10.0,balanced,lbfgs,mean,"{'model__C': 10.0, 'model__class_weight': 'bal...",0.463415,...,0.481977,0.019125,1,0.512195,0.480938,0.488095,0.505952,0.498462,0.497129,0.011405
23,0.007720,0.000476,0.003522,0.000467,10.0,balanced,liblinear,mean,"{'model__C': 10.0, 'model__class_weight': 'bal...",0.463415,...,0.481977,0.019125,1,0.512195,0.480938,0.488095,0.505952,0.498462,0.497129,0.011405
20,0.008932,0.000576,0.005002,0.000759,10.0,balanced,lbfgs,median,"{'model__C': 10.0, 'model__class_weight': 'bal...",0.457831,...,0.480861,0.020303,3,0.510638,0.480938,0.488095,0.505952,0.498462,0.496817,0.011004
22,0.008235,0.000629,0.004610,0.000556,10.0,balanced,liblinear,median,"{'model__C': 10.0, 'model__class_weight': 'bal...",0.457831,...,0.480861,0.020303,3,0.510638,0.480938,0.488095,0.505952,0.498462,0.496817,0.011004
13,0.008885,0.000503,0.004625,0.000174,1.0,balanced,lbfgs,mean,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.463415,...,0.479903,0.020609,5,0.515152,0.477876,0.488095,0.502959,0.504615,0.497739,0.013155
12,0.010868,0.001727,0.005188,0.000839,1.0,balanced,lbfgs,median,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.457831,...,0.478786,0.021599,6,0.512048,0.482353,0.488095,0.502959,0.504615,0.498014,0.011033
14,0.009044,0.000700,0.005254,0.001604,1.0,balanced,liblinear,median,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.457831,...,0.477417,0.019466,7,0.510511,0.482353,0.486647,0.502959,0.501529,0.496800,0.010584
15,0.007999,0.000401,0.004637,0.000192,1.0,balanced,liblinear,mean,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.457831,...,0.477417,0.019466,7,0.510511,0.482353,0.486647,0.502959,0.501529,0.496800,0.010584


In [26]:
print("GB best CV f1:", round(grid_gb.best_score_, 3))
print("LR best CV f1:", round(grid_lr.best_score_, 3))

winner_name = "LogisicRegression"
winner = grid_lr.best_estimator_

print("Winner:", winner_name)

GB best CV f1: 0.385
LR best CV f1: 0.482
Winner: LogisicRegression


In [27]:
test_pred = winner.predict(X_test)

print("Confusion matrix\n", confusion_matrix(y_test, test_pred))
print("\nClassification report:\n", classification_report(y_test, test_pred))

Confusion matrix
 [[206  54]
 [  8  32]]

Classification report:
               precision    recall  f1-score   support

           0       0.96      0.79      0.87       260
           1       0.37      0.80      0.51        40

    accuracy                           0.79       300
   macro avg       0.67      0.80      0.69       300
weighted avg       0.88      0.79      0.82       300



In [29]:
test_proba = winner.predict_proba(X_test)[:, 1]

ranked = X_test.copy()
ranked["p_pos"] = test_proba
ranked["y_true"] = y_test.values

top10 = ranked.sort_values("p_pos", ascending=False).head(10)

top10

,age,income,tenure_months,city,plan,channel,p_pos,y_true
447,67.0,9704.559899,5.0,Stockholm,basic,web,0.968485,1
997,36.0,17556.808641,3.0,Malmö,basic,web,0.967896,0
373,37.0,14318.439111,18.0,Göteborg,basic,partner,0.966639,1
1123,30.0,21464.563718,1.0,Göteborg,basic,web,0.962069,1
654,46.0,21526.778813,4.0,Stockholm,basic,web,0.933877,0
918,31.0,21752.136376,2.0,Stockholm,basic,store,0.920601,1
37,61.0,NaN,3.0,Göteborg,basic,web,0.916357,1
1199,50.0,18838.369415,23.0,Stockholm,basic,partner,0.915184,0
274,39.0,29122.997805,7.0,Malmö,basic,store,0.900708,1
585,66.0,31944.701963,1.0,Stockholm,basic,web,0.897640,1


In [30]:
k = 20
precision_at_k = ranked.sort_values("p_pos", ascending=False).head(k)["y_true"].mean()

print(f"Precision@{k}:", round(float(precision_at_k), 3))

Precision@20: 0.55


In [31]:
oof_proba = cross_val_predict(
    winner,
    X_train, y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
f1s = []
for t in thresholds:
    preds = (oof_proba >= t).astype(int)
    f1s.append(f1_score(y_train, preds))

best_t = float(thresholds[int(np.argmax(f1s))])

print("Best threshold (OOF, max F1):", round(best_t, 3))
print("Best OOF F1:", round(float(max(f1s)), 3))

Best threshold (OOF, max F1): 0.67
Best OOF F1: 0.511


In [35]:
test_pred_tuned = (test_proba >= best_t).astype(int)

print("Confusion matrix\n", confusion_matrix(y_test, test_pred_tuned))
print("\nClassification report:\n", classification_report(y_test, test_pred_tuned))

Confusion matrix
 [[234  26]
 [ 16  24]]

Classification report:
               precision    recall  f1-score   support

           0       0.94      0.90      0.92       260
           1       0.48      0.60      0.53        40

    accuracy                           0.86       300
   macro avg       0.71      0.75      0.73       300
weighted avg       0.88      0.86      0.87       300



In [36]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def pos_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average="binary", zero_division=0
    )
    return float(p), float(r), float(f)

p0, r0, f0 = pos_metrics(y_test, test_pred)
p1, r1, f1 = pos_metrics(y_test, test_pred_tuned)

summary = pd.DataFrame([
    {"setting": "Default threshold (0.5)", "precision(1)": p0, "recall(1)": r0, "f1(1)": f0, "accuracy": accuracy_score(y_test, test_pred)},
    {"setting": f"Tuned threshold ({best_t:.2f})", "precision(1)": p1, "recall(1)": r1, "f1(1)": f1, "accuracy": accuracy_score(y_test, test_pred_tuned)},
])
summary

,setting,precision(1),recall(1),f1(1),accuracy
0,Default threshold (0.5),0.372093,0.8,0.507937,0.793333
1,Tuned threshold (0.67),0.480000,0.6,0.533333,0.860000
